# InsightCrew — Multi-Agent Data Analysis (on NVIDIA NOOA)

A crew of LLM agents plans the analysis and writes the report; a deterministic pandas
engine computes the real numbers and charts. Figures are always correct — never
hallucinated. Runs **free** on the NVIDIA NIM API (no GPU needed).

Every agent makes a single structured call, so the whole run finishes in seconds even
on the small free `llama-3.1-8b` model.


## 1. Upload and unzip the project
Run this, then choose `insightcrew.zip` when the upload button appears.


In [ ]:
from google.colab import files
files.upload()            # choose insightcrew.zip
!unzip -oq insightcrew.zip
%cd insightcrew


## 2. Install dependencies (~2 min)


In [ ]:
!pip install -q nooa pandas matplotlib python-dotenv


## 3. Set your free NVIDIA key
Get one at https://build.nvidia.com. `getpass` keeps it out of the saved notebook.


In [ ]:
import os, getpass
os.environ['NOOA_PROVIDER'] = 'nvidia'
os.environ['NVIDIA_MODEL']  = 'nvidia_nim/meta/llama-3.1-8b-instruct'  # fast + free
os.environ['NVIDIA_API_KEY'] = getpass.getpass('NVIDIA API key (nvapi-...): ').strip()
os.environ['NVIDIA_NIM_API_KEY'] = os.environ['NVIDIA_API_KEY']  # LiteLLM looks for this


## 4. Run the crew
Plan → compute (deterministic) → critique → report. Colab allows top-level `await`.


In [ ]:
from insightcrew.orchestrator import InsightCrew
from insightcrew.llm_setup import build_llm

crew = InsightCrew('data/sample_sales.csv', llm=build_llm(), charts_dir='charts')
report = await crew.run('Which region and category drive our revenue, and what changed over the year?')
print(report.headline, '|', report.confidence)


## 5. Read the report and charts
The numbers here come from pandas, so they match the charts exactly.


In [ ]:
from IPython.display import Markdown, Image, display
import glob
display(Markdown(report.body_markdown))
for path in sorted(glob.glob('charts/*.png')):
    print(path); display(Image(path))


## 6. Ask your own question
Try anything the data supports — by channel, by segment, by product, over time.


In [ ]:
report = await crew.run('Compare Online vs Retail revenue, and which segment spends more per unit.')
display(Markdown(report.body_markdown))
for path in sorted(glob.glob('charts/*.png')):
    display(Image(path))
